## 0. Introduction
This notebook demontrates the core functionalities of **NumCompute**, a lightweight and fully vectorized machine learning framework built from scratch using Numpy.

In [4]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

## 1. CSV Data Loading
In this section, we demonstrate how to load a dataset containing missing values from a CSV file using NumCompute's custom `io` module. If the file is missing, a synthetic fallback dataset is generated to ensure seamless execution.

In [5]:
import numpy as np 
from numcompute_stream.io import load_csv

#Loading a dataset using out CSV Loader 
print("---- 1. Testing CSV Loader ----")
try:
    data = load_csv("data.csv", delimiter=",", skiprows=1)
    print("Data loaded successfully from the .csv file! Shape:", data.shape)
except Exception as e:
    #Returning fallback dummy dataset if data.csv is not found
    print("Error while loading: ", e)
    print("Creating fallback dataset with missing values...")
    data = np.array([
        [1.0, np.nan, 2.0],
        [3.0, 4.0, np.nan],
        [5.0, 6.0, 7.0],
        [1.0, 4.0, 2.0]
        ])
    print("Synthetic data shape:", data.shape)



---- 1. Testing CSV Loader ----
Data loaded successfully from the .csv file! Shape: (4, 3)


## 2. Data Preprocessing
We utilize our vectorized transformers (`SimpleImputer`, `StandardScaler`, `MinMaxScaler`, and `OneHotEncoder`) to clean and scale the dataset without Python loops.

In [6]:
from numcompute_stream.preprocessing import SimpleImputer, StandardScaler, MinMaxScaler, OneHotEncoder

print("\n---- 2. Testing Preprocessing Transformers ----")

#Handle missing values using mean imputation
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(data)
print("After Imputation:\n", X_imputed)

#Scale features to zero mean and unit variance using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
print("\nAfter StandardScaler:\n", X_scaled)

#Categorical encoding using OneHotEncoder
categorical_data = np.array([[0], [1], [2], [0], [1]])
encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(categorical_data)
print("\nAfter OneHotEncoder:\n", X_encoded)



---- 2. Testing Preprocessing Transformers ----
After Imputation:
 [[ 1.          6.33333333  2.        ]
 [ 3.          4.          6.33333333]
 [ 5.          6.          7.        ]
 [ 8.          9.         10.        ]]

After StandardScaler:
 [[-1.25675744  0.         -1.51635079]
 [-0.48336824 -1.31122014  0.        ]
 [ 0.29002095 -0.18731716  0.23328474]
 [ 1.45010473  1.4985373   1.28306606]]

After OneHotEncoder:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


## 3. Chaining Steps with Pipeline
We can seamlessly chain multiple preprocessing steps together using the `Pipeline` class, following standard scikit-learn workflows.

In [7]:
from numcompute_stream.pipeline import Pipeline

print("\n---- 3. Testing Pipeline ----")

#Define a sequential pipeline
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy="mean")),
    ('scaler', MinMaxScaler(feature_range=(0.0, 1.0)))
])

#Fit and transform
X_pipeline_result = pipe.fit_transform(data)
print("Pipeline final output:\n", X_pipeline_result)

print("\nYour preprocessing and pipeline sections are successfully set up! You're ready for the next module.")



---- 3. Testing Pipeline ----
Pipeline final output:
 [[0.         0.46666667 0.        ]
 [0.28571429 0.         0.54166667]
 [0.57142857 0.4        0.625     ]
 [1.         1.         1.        ]]

Your preprocessing and pipeline sections are successfully set up! You're ready for the next module.


## 4. Sorting, Searching and Ranking
`sort_search` provides top-k selection (`np.argpartition`), quickselect and binary search; `rank` provides tie-aware ranking and percentiles. NaN is treated as missing: it is never selected before a real value, stays unranked, and is ignored by `percentile`.

In [8]:
from numcompute_stream.sort_search import topk, quickselect, binary_search
from numcompute_stream.rank import rank, percentile

print("\n---- 4. Testing Sorting, Searching and Ranking ----")

#Scores with a tie (95) and a missing value (NaN)
scores = np.array([72, 95, 88, 95, 60, np.nan, 81])

#Top-k: the two tied 95s keep their original order (index 1, then 3)
values, indices = topk(scores, 3)
print("Top-3 values:", values, "| indices:", indices)

#Quickselect: k-th smallest value (0-based) without sorting the whole array
print("3rd smallest (k=2):", quickselect(np.array([72, 95, 88, 95, 60, 81]), 2, seed=0))

#Binary search: insertion index and whether the value exists
index, found = binary_search(np.array([60, 72, 81, 88, 95]), [81, 90])
print("Binary search for [81, 90] -> index:", index, "| found:", found)

#Ranking with three tie-handling methods (NaN stays unranked)
for method in ["average", "dense", "ordinal"]:
    print(f"Rank ({method}):", rank(scores, method=method))

#Percentiles ignore NaN; four interpolation methods are supported
print("Percentiles (25, 50, 75):", percentile(scores, [25, 50, 75]))
print("Median (linear vs lower):", percentile(scores, 50), "vs", percentile(scores, 50, interpolation="lower"))

#Ranking the first column of the preprocessed dataset from section 2
print("\nRank of column 0 in X_imputed:", rank(X_imputed[:, 0], method="average"))


---- 4. Testing Sorting, Searching and Ranking ----
Top-3 values: [95. 95. 88.] | indices: [1 3 2]
3rd smallest (k=2): 81
Binary search for [81, 90] -> index: [2 4] | found: [ True False]
Rank (average): [2.  5.5 4.  5.5 1.  nan 3. ]
Rank (dense): [ 2.  5.  4.  5.  1. nan  3.]
Rank (ordinal): [ 2.  5.  4.  6.  1. nan  3.]
Percentiles (25, 50, 75): [74.25 84.5  93.25]
Median (linear vs lower): 84.5 vs 81.0

Rank of column 0 in X_imputed: [1. 2. 3. 4.]


## 5. Statistics, Metrics, Optimisation and Benchmarking
This section demonstrates descriptive statistics, model evaluation metrics, finite-difference derivatives, and the loop-versus-NumPy benchmark.

In [9]:
from numcompute_stream.stats import mean, median, std, histogram, quantiles
from numcompute_stream.metrics import accuracy, precision, recall, f1, confusion_matrix, mse
from numcompute_stream.optim import grad, jacobian

print("\n---- 5. Statistics ----")
print("Mean:", mean(scores))
print("Median:", median(scores))
print("Standard deviation:", std(scores))
print("Quantiles:", quantiles(scores, [0.25, 0.5, 0.75]))
counts, edges = histogram(scores, bins=3)
print("Histogram counts:", counts)

print("\n---- 5. Metrics ----")
y_true = np.array([1, 1, 0, 0, 1])
y_pred = np.array([1, 0, 0, 0, 1])
print("Accuracy:", accuracy(y_true, y_pred))
print("Precision / recall / F1:", precision(y_true, y_pred), recall(y_true, y_pred), f1(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
print("MSE:", mse([1, 2, 3], [1, 3, 2]))

print("\n---- 5. Finite differences ----")
quadratic = lambda x: np.sum(x ** 2)
print("Gradient of x^2 at [2, 3]:", grad(quadratic, np.array([2.0, 3.0])))
vector_function = lambda x: np.array([x[0] ** 2, x[0] * x[1]])
print("Jacobian at [2, 3]:\n", jacobian(vector_function, np.array([2.0, 3.0])))


---- 5. Statistics ----
Mean: 81.83333333333333
Median: 84.5
Standard deviation: 12.6414222125343
Quantiles: [74.25 84.5  93.25]
Histogram counts: [1 2 3]

---- 5. Metrics ----
Accuracy: 0.8
Precision / recall / F1: 1.0 0.6666666666666666 0.8
Confusion matrix:
 [[2 0]
 [1 2]]
MSE: 0.6666666666666666

---- 5. Finite differences ----
Gradient of x^2 at [2, 3]: [4. 6.]
Jacobian at [2, 3]:
 [[4. 0.]
 [3. 2.]]


In [10]:
from numcompute_stream.benchmarking import run_benchmarks, write_results

print("\n---- 5. Benchmark ----")
benchmark_rows = run_benchmarks(n=10000, repeats=5, seed=42)
write_results(benchmark_rows, "../benchmark", 10000, 5, 42)
print("Operation | loop ms | vectorised ms | speed-up")
for name, n, loop_ms, vector_ms, speedup in benchmark_rows:
    print(f"{name}: {loop_ms:.3f} | {vector_ms:.3f} | {speedup:.1f}x")


---- 5. Benchmark ----
Operation | loop ms | vectorised ms | speed-up
StandardScaler: 3.585 | 0.015 | 243.8x
top-k: 1.754 | 0.202 | 8.7x
rank: 3.337 | 0.575 | 5.8x
histogram: 1.152 | 0.089 | 12.9x
